In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
pd.set_option('display.max_columns', 100)

In [ ]:
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv', parse_dates=['date'])
sample_submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("\nTrain info:")
train.info()
print("\nTest info:")
test.info()

In [ ]:
oil['dcoilwtico'] = oil['dcoilwtico'].replace(0, np.nan)
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear')

In [ ]:
holidays['is_holiday'] = holidays['type'].apply(lambda x: 1 if x in ['Holiday', 'Transfer', 'Bridge', 'Additional'] else 0)
holidays['is_transferred'] = holidays['transferred'].apply(lambda x: 1 if x == True else 0)

transferred_dates = holidays[holidays['is_transferred'] == 1]['date']
holidays.loc[holidays['date'].isin(transferred_dates), 'is_holiday'] = 0

holidays_consolidated = holidays.groupby('date').agg({
    'is_holiday': 'max',
    'is_transferred': 'max',
    'locale': lambda x: ','.join(x),
    'type': lambda x: ','.join(x)
}).reset_index()

In [ ]:
def add_date_features(df):
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    return df

train = add_date_features(train)
test = add_date_features(test)

In [ ]:
def merge_data(df):
    # Merge with stores data
    df = pd.merge(df, stores, on='store_nbr', how='left', suffixes=('', '_store'))
    
    # Merge with oil data
    df = pd.merge(df, oil, on='date', how='left')
    
    # Merge with holidays data
    df = pd.merge(df, holidays_consolidated, on='date', how='left', suffixes=('', '_holiday'))
    
    # Fill missing holiday values
    df['is_holiday'] = df['is_holiday'].fillna(0)
    df['is_transferred'] = df['is_transferred'].fillna(0)
    df['locale'] = df['locale'].fillna('None')
    
    if 'type_holiday' in df.columns:
        df['type_holiday'] = df['type_holiday'].fillna('None')
    
    return df

train = merge_data(train)
test = merge_data(test)

In [ ]:
cat_cols = ['family', 'city', 'state', 'type_holiday', 'locale', 'type']
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train[col], test[col]], axis=0))
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])

In [ ]:
train.head()

In [ ]:
# 1. Sales distribution
plt.figure(figsize=(12, 6))
sns.histplot(train['sales'], bins=50, kde=True)
plt.title('Sales Distribution')
plt.xlabel('Sales')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 2. Sales over time
plt.figure(figsize=(16, 8))
train.groupby('date')['sales'].sum().plot()
plt.title('Total Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Total Sales')
plt.show()

In [ ]:
# 3. Sales by store type
plt.figure(figsize=(12, 6))
sns.boxplot(x='type', y='sales', data=train)
plt.title('Sales Distribution by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Sales')
plt.show()

In [ ]:
# 4. Oil price vs sales
plt.figure(figsize=(12, 6))
sns.scatterplot(x='dcoilwtico', y='sales', data=train.sample(10000))
plt.title('Oil Price vs Sales')
plt.xlabel('Oil Price')
plt.ylabel('Sales')
plt.show()

In [ ]:
# 5. Sales by day of week
plt.figure(figsize=(12, 6))
sns.boxplot(x='day_of_week', y='sales', data=train)
plt.title('Sales by Day of Week')
plt.xlabel('Day of Week (0=Monday)')
plt.ylabel('Sales')
plt.show()

In [ ]:
# 6. Holiday impact on sales
plt.figure(figsize=(12, 6))
sns.boxplot(x='is_holiday', y='sales', data=train)
plt.title('Holiday Impact on Sales')
plt.xlabel('Is Holiday')
plt.ylabel('Sales')
plt.show()

In [ ]:
# 7. Sales by product family
plt.figure(figsize=(16, 8))
train.groupby('family')['sales'].sum().sort_values(ascending=False).plot(kind='bar')
plt.title('Total Sales by Product Family')
plt.xlabel('Product Family')
plt.ylabel('Total Sales')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Time Series Decomposition for a sample store and product
sample_ts = train[(train['store_nbr'] == 1) & (train['family'] == 1)].set_index('date')['sales']
sample_ts_daily = sample_ts.resample('D').mean().interpolate(method='time')
decomposition = seasonal_decompose(sample_ts_daily, model='additive', period=365)

plt.figure(figsize=(16, 12))
plt.subplot(411)
decomposition.observed.plot()
plt.title('Observed')
plt.subplot(412)
decomposition.trend.plot()
plt.title('Trend')
plt.subplot(413)
decomposition.seasonal.plot()
plt.title('Seasonal')
plt.subplot(414)
decomposition.resid.plot()
plt.title('Residual')
plt.tight_layout()
plt.show()

In [ ]:
features = ['store_nbr', 'family', 'onpromotion', 'year', 'month', 'day', 
            'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 
            'is_weekend', 'is_month_start', 'is_month_end', 'city', 
            'state', 'type', 'cluster', 'dcoilwtico', 'is_holiday', 
            'is_transferred']

X = train[features]
y = train['sales']

X_test = test[features]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lgb_params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'n_estimators': 1000,
    'importance_type': 'gain',
    'verbose': -1
}

lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    valid_sets=[lgb_train, lgb_val]
)

In [ ]:
plt.figure(figsize=(12, 8))
lgb.plot_importance(lgb_model, max_num_features=10)
plt.title('Feature Importance')
plt.show()

In [ ]:
test_preds = lgb_model.predict(X_test)

In [ ]:
submission = test[['id']].copy()
submission['sales'] = test_preds
submission.to_csv('submission.csv', index=False)

In [ ]:
submission.head()